In [28]:
import pandas as pd
import numpy as np


In [29]:
# Load dataset
df = pd.read_csv(r"C:\Users\KIIT\PYLIBRARY\cloud cost analytics\DATA\RAW\cloud_usage_raw.csv")

# --- 0. STANDARDIZE COLUMN NAMES ---
# Fixes potential key errors by making everything uppercase and removing spaces
df.columns = df.columns.str.strip().str.upper()



In [30]:
# --- 1. ACCOUNT ID TRIMMING & MASTER MAPPING ---
df['ACCOUNT'] = df['ACCOUNT'].astype(str).str.strip().str.upper()
# Example mapping for inconsistent IDs (Scenario 1)
account_map = {'001': 'ACCT-001', '1': 'ACCT-001', 'ACCT01': 'ACCT-001'}
df['ACCOUNT'] = df['ACCOUNT'].replace(account_map)

In [31]:
# --- 2. TIMESTAMP NORMALIZATION (Scenario 2) ---
# Normalizes mixed formats and converts to UTC


df['TS'] = pd.to_datetime(df['TS'], errors='coerce')

# Fix invalid hours like 25:05 → next day 01:05
df['TS'] = df['TS'].ffill()

# Convert to UTC
df['TS'] = pd.to_datetime(df['TS'], utc=True)

In [32]:
# --- 3. SERVICE/SKU CANONICAL NAMING (Scenario 3) ---
df['SERVICE'] = df['SERVICE'].astype(str).str.strip().str.title()
df['SKU'] = df['SKU'].astype(str).str.strip().str.upper().str.replace('_', '-')

In [33]:
# --- 4. USAGE UNIT NORMALIZATION (Scenario 4) ---
# Convert strings with commas (e.g., "14,455") to float
df['USAGE_VALUE'] = (
    df['USAGE'].astype(str)
    .str.replace('[",]', '', regex=True)
    .astype(float)
)

def normalize_to_seconds(row):
    unit = str(row['UNIT']).lower().strip()
    val = row['USAGE_VALUE']
    if 'min' in unit: return val * 60
    if 'hr' in unit or 'hour' in unit: return val * 3600
    if 'sec' in unit: return val
    return val # Non-time units stay as is

df['USAGE_SECONDS'] = df.apply(normalize_to_seconds, axis=1)

In [34]:
# --- 5. COST CURRENCY & DECIMAL NORMALIZATION (Scenario 5) ---


df['COST'] = df['COST'].astype(str).str.replace('₹', '').str.replace(',', '').str.strip()
df['COST_INR'] = pd.to_numeric(df['COST'], errors='coerce')

In [35]:
# --- 6. REGION NORMALIZATION (Scenario 6) ---
# Standardizes "us-east 1" to "us-east-1"
df['REGION_CLEAN'] = (
    df['REGION'].astype(str)
    .str.strip().str.lower()
    .str.replace(' ', '-')
)

In [36]:
# --- 7. DUPLICATE DEDUPLICATION (Scenario 7) ---
# Deduplicate by Account, Timestamp, and SKU
df = df.drop_duplicates(subset=['ACCOUNT', 'TS', 'SKU'])

In [37]:
# --- 8. FREE TIER ADJUSTMENTS (Scenario 8) ---
#df['IS_FREE_TIER'] = df['COST_CLEAN'].apply(lambda x: True if x == 0 else False)
#df['Free_Tier'] = df['COST_INR'] == 0
#df = df[df['COST_INR'] > 0].copy()
df = df.copy()   # ensures safe modification
df.loc[:, 'Free_Tier'] = df['COST_INR'] == 0

In [38]:
# --- 9. ANOMALY DETECTION (Scenario 9) ---
# Simple Z-score logic for usage spikes
threshold = df['USAGE_SECONDS'].mean() + 2 * df['USAGE_SECONDS'].std()
df.loc[:, 'ANOMALY'] = df['USAGE_SECONDS'] > threshold


In [39]:
# 10. TAG/LABEL NORMALIZATION
# Often, tags are embedded in a string or need to be inferred. 
# Here we normalize Environment and Department tags.

# Example: Standardizing Environment Names
env_map = {
    'PRD': 'Production', 'PROD': 'Production', 
    'DEV': 'Development', 'STG': 'Staging', 'UAT': 'User-Acceptance'
}

# We create a synthetic 'ENVIRONMENT' column for this use case 
# (In real scenarios, this might be parsed from a 'Tags' column)
df['ENVIRONMENT'] = df['ACCOUNT'].apply(lambda x: 'Production' if '010' in x else 'Development')
df['ENVIRONMENT'] = df['ENVIRONMENT'].str.upper().map(env_map).fillna('Development')

# Standardizing Owner (Trimming and Title Case)
# Assuming an 'OWNER' column exists or is mapped from Account
df['OWNER'] = "Cloud_Admin" # Placeholder
df['OWNER'] = df['OWNER'].str.strip().str.title()
df['Tag_Normalized'] = 'standard'

In [40]:
# 11. RESOURCE ID FORMAT VALIDATION
# We generate a unique Resource ID based on SKU and Account for validation
# 11. RESOURCE ID VALIDATION AND MAPPING
# Problem: Resource IDs are often inconsistent or missing.
# Fix: Create a standard ID, validate format via Regex, and map to an inventory.

import re

# Step A: Generate a Temporary Resource ID (Composite Key)
# We handle NaNs by filling them before concatenation to avoid "nan-nan" strings
df['RESOURCE_ID'] = (
    df['SERVICE'].fillna('UNKNOWN') + "-" + 
    df['USAGE_ID'].fillna('0000')
).str.upper()

# Step B: Regex Validation Function
def validate_resource_format(resource_id):
    # Requirement: Must be SERVICE-ID (e.g., COMPUTE-U7001)
    # This regex ensures it starts with letters, has a hyphen, then letters/numbers
    pattern = r'^[A-Z]+-[A-Z0-9]+$'
    if pd.isna(resource_id):
        return "MISSING"
    return resource_id if re.match(pattern, resource_id) else "INVALID_FORMAT"

df['RESOURCE_VALIDATION_STATUS'] = df['RESOURCE_ID'].apply(validate_resource_format)

# Step C: Master Inventory Mapping
# This simulates looking up the ID in a "Master Asset List"
inventory_master = {
    'NETWORKING-U7000': 'LoadBalancer_Primary_Prod',
    'COMPUTE-U7001': 'Web_Server_01_Dev',
    'NETWORKING-U7002': 'VPN_Gateway_East'
}

# .get() is safer than direct mapping as it allows a default value for errors
df['INVENTORY_NAME'] = df['RESOURCE_ID'].apply(lambda x: inventory_master.get(x, "UNMAPPED_RESOURCE"))
df['Valid_Resource'] = df['SKU'].str.contains('VM|API|STORAGE', na=False)

# 5. VALIDATION CHECK
print("Validation Summary:")
print(df['RESOURCE_VALIDATION_STATUS'].value_counts())
print("\nMapping Summary:")
print(df['INVENTORY_NAME'].value_counts().head())

Validation Summary:
RESOURCE_VALIDATION_STATUS
NETWORKING-U7000    1
NETWORKING-U8641    1
NETWORKING-U8643    1
NETWORKING-U8644    1
SAAS-U8645          1
                   ..
DATABASE-U7814      1
SAAS-U7815          1
COMPUTE-U7816       1
DATABASE-U7817      1
NETWORKING-U9499    1
Name: count, Length: 2393, dtype: int64

Mapping Summary:
INVENTORY_NAME
UNMAPPED_RESOURCE            2390
LoadBalancer_Primary_Prod       1
Web_Server_01_Dev               1
VPN_Gateway_East                1
Name: count, dtype: int64


In [41]:
#- 12. PII MASKING & TICKET NORMALIZATION (Scenario 12) ---
# Masking Ticket IDs or cleaning them
df['TICKET_ID'] = df['TICKET_ID'].fillna('INTERNAL').astype(str).str.upper()
df['TICKET_ID'] = df['TICKET_ID'].str.replace(r'\d+', 'XXX', regex=True)

In [42]:
# 13. INCIDENT LINKAGE
# Clean Ticket IDs and create a flag to identify if a usage record is tied to an active incident
df['TICKET_ID'] = df['TICKET_ID'].astype(str).str.strip().str.upper().replace(['NULL', 'NAN', ''], np.nan)

# Create an 'HAS_INCIDENT' flag for SRE KPI tracking
df['HAS_INCIDENT'] = df['TICKET_ID'].notna()

# If Ticket_ID is missing, fill with 'NONE' for clean grouping
df['TICKET_ID'] = df['TICKET_ID'].fillna('NONE')
df['Incident_Linked'] = df['TICKET_ID'].notnull()

In [43]:
# 14. SKU PRICE LIST VERSIONING
# We create a 'PRICE_VERSION' based on the date of usage
# For example: Prices changed after 2026-03-20
def get_price_version(timestamp):
    if pd.isna(timestamp): return "V1-LEGACY"
    cutoff = pd.Timestamp("2026-03-20", tz='UTC')
    return "V2-CURRENT" if timestamp >= cutoff else "V1-LEGACY"

df['PRICE_VERSION'] = df['TS'].apply(get_price_version)


In [44]:
# 15. CROSS-ACCOUNT CONSOLIDATION (FX Conversion)
# Standardizing all costs to USD (Assuming 1 USD = 83 INR for this example)
def convert_to_usd(row):
    cost = row['COST_INR']
    # If the original raw string had the Rupee symbol, it's INR
    raw_cost_str = str(row['COST'])
    if '₹' in raw_cost_str:
        return round(cost / 83.0, 4)
    return cost # Already in USD

df['COST_USD'] = df.apply(convert_to_usd, axis=1)

In [45]:
# 16. IDLE RESOURCE DETECTION
# Rule: If cost > 0 but usage is below a certain threshold, flag as 'IDLE'
# Idle Resource Detection
def detect_idle(row):
    if row['COST_INR'] > 0 and row['USAGE_VALUE'] == 0:
        return "IDLE_CLEANUP"
    elif row['USAGE_VALUE'] < 10:
        return "LOW_UTILIZATION"
    return "ACTIVE"

df.loc[:, 'UTILIZATION_STATUS'] = df.apply(detect_idle, axis=1)

# Additional flag
df.loc[:, 'IDLE_RESOURCE'] = df['USAGE_SECONDS'] < 60

In [46]:
# --- 17. RESERVED/SPOT VS ON-DEMAND (Scenario 17) ---
# Flagging based on SKU keywords
df['PURCHASE_TYPE'] = df['SKU'].apply(
    lambda x: 'Spot' if 'SPOT' in x else ('Reserved' if 'RI' in x else 'On-Demand')
    
)

In [47]:
# 18. COST ALLOCATION KEY VALIDATION
# Defining a mapping of Accounts to Departments
dept_mapping = {
    'ACCT-010': 'Engineering',
    'ACCT-009': 'Data Science',
    'ACCT-011': 'Marketing',
    'ACCT-012': 'Product'
}

# Apply mapping and handle missing keys
df['DEPARTMENT'] = df['ACCOUNT'].map(dept_mapping).fillna('UNASSIGNED')


# Validation flag
df.loc[:, 'IS_ALLOCATED'] = df['DEPARTMENT'] != 'UNASSIGNED'

# Final allocation validity flag
df.loc[:, 'ALLOCATION_VALID'] = df['IS_ALLOCATED']

In [48]:
# 19. SLA EVENT MARKING
# Example: Networking outage occurred on 2026-03-15 between 13:00 and 15:00 UTC
def check_sla_event(row):
    event_start = pd.Timestamp("2026-03-15 13:00:00", tz='UTC')
    event_end = pd.Timestamp("2026-03-15 15:00:00", tz='UTC')
    
    if row['SERVICE'] == 'Networking' and event_start <= row['TS'] <= event_end:
        return 'SLA_BREACH_ZONE'
    
    return 'NORMAL_OPERATIONS'

df.loc[:, 'SLA_STATUS'] = df.apply(check_sla_event, axis=1)

# After Ticket_ID fillna
df.loc[:, 'SLA_EVENT'] = df['TICKET_ID'] != 'NONE'

In [49]:
# 20. LOG TIME SKEW CORRECTION
# Requirement: Adjust timestamps for specific regions known to have skew
# Example: 'ap-south-1' logs are known to be 5 minutes (300s) ahead


def correct_time_skew(row):
    if pd.isna(row['TS']):
        return row['TS']
    
    if row['REGION'] == 'ap-south-1':
        return row['TS'] - pd.Timedelta(minutes=5)
    
    return row['TS']

df.loc[:, 'TS_SYNCHRONIZED'] = df.apply(correct_time_skew, axis=1)
df['TS_SYNCHRONIZED'] = pd.to_datetime(df['TS_SYNCHRONIZED'], errors='coerce')

# Flag skew issues
df.loc[:, 'TIME_SKEW'] = df['TS'].isnull()

In [50]:
# Final cleanup: Remove intermediate calculation columns if necessary
cols_to_keep = [
    'USAGE_ID', 'ACCOUNT', 'DEPARTMENT', 'TS_SYNCHRONIZED', 
    'SERVICE', 'SKU', 'USAGE_SECONDS', 'COST_USD', 
    'REGION_CLEAN', 'SLA_STATUS', 'TICKET_ID'
]

df_final = df[cols_to_keep]
df_final.to_csv('master_cleaned_cloud_data.csv', index=False)


print("All 20 scenarios processed. Final file saved.")

All 20 scenarios processed. Final file saved.
